## Communication Topology and Belief Dynamics in Multi-Agent LLM Reasoning
### Experiment Analysis & Visualisation

This notebook analyses experimental results from a multi-agent LLM system tested across different communication topologies on the GSM8K benchmark.

**Topologies tested:**
- **Independent**: agents reason alone, answers aggregated via majority vote
- **Fully Connected**: all agents see each other's responses before revising
- **Mediator**: a mediator summarises responses; agents see only the summary
- **Chain**: agents answer sequentially, each seeing only the previous agent

**Primary questions:**
1. Does collaboration improve accuracy over independent reasoning?
2. How do different communication structures affect convergence?
3. What are the cost/accuracy trade-offs across topologies?

In [17]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import pandas as pd
import numpy as np
import warnings
import glob
from pathlib import Path
warnings.filterwarnings("ignore")

---
### 1. Setup & Data Loading

In [18]:
pio.templates.default = "plotly_white"

TOPO_COLORS = {
    "independent": "#636EFA",
    "fully_connected": "#EF553B",
    "mediator": "#00CC96",
    "chain": "#AB63FA",
}

AGENT_COLORS = {
    "gemma3:4b": "#FF6B6B",
    "phi4-mini": "#4ECDC4",
    "llama3.2:3b": "#45B7D1",
    "qwen2.5:3b-instruct": "#F7DC6F",
}

AGENT_NAME_MAP = {
    0: "gemma3:4b", 
    1: "phi4-mini", 
    2: "llama3.2:3b", 
    3: "qwen2.5:3b-instruct"
}

In [ ]:
# Load and concatenate all results into a single DataFrame
result_files = list(Path("../results/run_1").glob("*.csv"))
print(f"Found {len(result_files)} result files:")
for f in result_files:
    print(f"  {f}")

df = pd.concat((pd.read_csv(f) for f in result_files), ignore_index=True)

Found 4 result files:
  ..\results\chain_20260313_043656.csv
  ..\results\full_20260312_224144.csv
  ..\results\independent_20260312_212352.csv
  ..\results\mediator_20260313_074928.csv


In [27]:
# Quick data overview
print("=== Dataset Summary ===")
print(f"Shape: {df.shape}")
print("\nTopology counts:")
print(df['topology'].value_counts())
print("\nRounds per topology:")
print(df.groupby('topology')['round'].max())
print("\nTemperature:        0.4 \nSamples questions:  200")
print(f"\nParse failure rate: {df['parse_failed'].mean():.2%}")

parse_by_model = df.groupby(["topology", "model"])["parse_failed"].mean().unstack()
print(parse_by_model.applymap(lambda x: f"{x:.1%}"))

=== Dataset Summary ===
Shape: (8800, 16)

Topology counts:
topology
chain          3200
full           2400
mediator       2400
independent     800
Name: count, dtype: int64

Rounds per topology:
topology
chain          4
full           3
independent    1
mediator       3
Name: round, dtype: int64

Temperature:        0.4 
Samples questions:  200

Parse failure rate: 6.69%
model       gemma3:4b llama3.2:3b phi4-mini qwen2.5:3b-instruct
topology                                                       
chain            5.9%       19.2%      2.5%                2.2%
full             3.3%       15.3%      0.5%                1.3%
independent      6.0%       39.5%      1.0%                0.0%
mediator         4.0%       16.3%      1.0%                1.0%


---
### 2. Accuracy Comparison Across Topologies

The fundamental question: **does collaboration help?**
We compare group-level accuracy (majority vote) across all topologies.

In [21]:
# Question-level accuracy table (one row per question per topology)
q_level = (
    df.groupby(["topology", "question_idx"])
    .agg(
        correct=("correct", "first"),
        expected=("expected_answer", "first"),
        group_answer=("group_answer", "first"),
        last_round_idx=("round", "idxmax"),
    )
    .reset_index()
)

last_round = df.loc[q_level["last_round_idx"]]

In [22]:
accuracy = q_level.groupby("topology")["correct"].mean().reset_index()
accuracy.columns = ["topology", "accuracy"]

In [23]:
fig = px.bar(
    accuracy,
    x="topology", y="accuracy",
    color="topology", color_discrete_map=TOPO_COLORS,
    text=accuracy["accuracy"].apply(lambda x: f"{x:.1%}"), 
    title="Group Accuracy by Communication Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology"},
)
fig.update_traces(textposition="outside")
# fig.show()

fig.write_image("figures/accuracy_by_topology.png", scale=2)

#### Individual Agent Accuracy vs Group Accuracy

Does (majority) voting actually help? Comparing individual agent accuracy to the group's majority-vote accuracy.

In [28]:
# All rows with max round per group (one per agent)
max_rounds = df.groupby(["topology", "question_idx"])["round"].max().reset_index()
max_rounds.columns = ["topology", "question_idx", "max_round"]

last_round = df.merge(max_rounds, on=["topology", "question_idx"])
last_round = last_round[last_round["round"] == last_round["max_round"]]

# Calculate individual agent accuracy
individual_acc = (
    last_round.groupby(["topology", "agent_id"])
    .apply(lambda g: (g["answer"] == g["expected_answer"]).mean())
    .reset_index(name="accuracy")
)
individual_acc["agent_id"] = individual_acc["agent_id"].map(AGENT_NAME_MAP)

# Add group accuracy
group_acc = q_level.groupby("topology")["correct"].mean().reset_index()
group_acc["agent_id"] = "Group Vote"
group_acc.columns = ["topology", "accuracy", "agent_id"]

# Combining individual and group accuracy
combined = pd.concat([individual_acc, group_acc], ignore_index=True)

agent_color_map = AGENT_COLORS.copy()
agent_color_map["Group Vote"] = "#000000"

fig = px.bar(
    combined, x="topology", y="accuracy",
    color="agent_id", barmode="group",
    title="Individual Agent vs Group Accuracy by Topology",
    labels={"accuracy": "Accuracy", "topology": "Topology", "agent_id": ""},
    text=combined["accuracy"].apply(lambda x: f"{x:.1%}"),
    category_orders={"topology": ["independent", "full", "mediator", "chain"]},
    color_discrete_map=agent_color_map
)

fig.update_traces(textposition="outside")

fig.show()
fig.write_image("figures/individual_vs_group_accuracy.png", scale=2, width=1500)

---
### 3. Confidence Analysis

How confident are agents, and does confidence correlate with correctness?
Over-confidence on wrong answers is a known LLM issue.